# Smart E-Commerce Customer Intelligence & Purchase Prediction System
## ML Lab Phase I Notebook: Data Preprocessing, EDA, Session Segmentation, ML Models & SHAP Explainability

**Dataset:** UCI Online Shoppers Purchasing Intention Dataset  
**Objective:** Analyze shopping behavior, segment visitor sessions, predict purchase intention (`Revenue`), and derive actionable business insights.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add root project directory to path
sys.path.append('..')

from src.preprocessing import load_raw_data, clean_data, get_feature_lists, build_preprocessor, prepare_data_split
from src.feature_engineering import create_session_features
from src.segmentation import evaluate_kmeans_range, evaluate_gmm_range, evaluate_dbscan, profile_clusters
from src.models import get_base_models
from src.evaluation import compare_all_models
from src.explainability import compute_shap_explainer, get_global_feature_importance

print("All modules imported successfully!")

### 1. Load Raw Dataset & Inspect Data Quality

In [ ]:
data_path = os.path.join('..', 'data', 'online_shoppers_intention.csv')
df_raw = load_raw_data(data_path)
print("Shape:", df_raw.shape)
print("Missing Values:", df_raw.isnull().sum().sum())
print("Duplicate Count:", df_raw.duplicated().sum())
df_raw.head()

### 2. Feature Engineering & Preprocessing

In [ ]:
df_feat = create_session_features(df_raw)
df_clean = clean_data(df_feat, drop_duplicates=True)
print("Processed Dataset Shape:", df_clean.shape)
df_clean[['Total_Page_Views', 'Total_Duration', 'Avg_Duration_Per_Page', 'PageValue_Intensity']].describe()

### 3. Session Segmentation (K-Means, GMM, DBSCAN)

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols_cluster = [
    'Administrative', 'Administrative_Duration', 'Informational',
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'Total_Page_Views',
    'Total_Duration', 'Avg_Duration_Per_Page', 'PageValue_Intensity'
]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[num_cols_cluster])

km_res = evaluate_kmeans_range(X_scaled)
gmm_res = evaluate_gmm_range(X_scaled)
db_res = evaluate_dbscan(X_scaled)

eval_all = pd.concat([km_res, gmm_res, db_res], ignore_index=True).drop(columns=['Model', 'Labels']).dropna()
eval_all

### 4. Machine Learning Classification & Model Comparison

In [ ]:
num_cols, cat_cols = get_feature_lists()
preprocessor = build_preprocessor(num_cols, cat_cols)
X_train, X_test, y_train, y_test = prepare_data_split(df_clean, target_col='Revenue', test_size=0.2, random_state=42)

pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
models_dict = get_base_models(scale_pos_weight=pos_weight)

comp_df, cm_dict, curves_dict, fitted_pipes = compare_all_models(models_dict, preprocessor, X_train, y_train, X_test, y_test)
comp_df

### 5. SHAP Model Explainability

In [ ]:
best_model_name = comp_df.iloc[0]['Model']
best_pipe = fitted_pipes[best_model_name]
explainer, shap_values, sample_df, feature_names = compute_shap_explainer(best_pipe, X_train, sample_size=300)
global_imp = get_global_feature_importance(shap_values, feature_names)
global_imp.head(10)